# 🏋️ Train your from-scratch model on a FREE GPU

This trains a ~20M-parameter model (6 layers, 512-token context) on dozens of
public-domain books + your own URLs. From **random weights** — no pretrained
models, no AI APIs. The result runs on a phone in <100 MB RAM.

**Before running:** menu **Runtime → Change runtime type → T4 GPU → Save.**

Then press ▶ on each step in order. Total time: roughly 3–4 hours
(mostly Step 4). You can re-run Step 4 again and again later — it resumes
from the last checkpoint, so the model keeps training *more and more*.

**Honest expectations:** after a few hours you get GPT-2-nano-class English —
fluent-ish sentences, story continuation, simple chat format. NOT ChatGPT.
ChatGPT-level needs millions of dollars of compute; nobody gets that from
scratch on free hardware. But this will be visibly, genuinely smarter than
the 25-minute CPU model, and it is 100% yours.

In [ ]:
#@title Step 1 — get the code, check the GPU { display-mode: "form" }
import os, sys, torch
REPO = "https://github.com/debzitsu-ship-it/Project-lmarena.git"
BRANCH = "arena/01a0011c-project-lmarena"
if not os.path.exists("Project-lmarena"):
    !git clone -q -b {BRANCH} {REPO}
%cd -q /content/Project-lmarena
sys.path.insert(0, "/content/Project-lmarena")
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
print("✅ GPU:", torch.cuda.get_device_name(0))

In [ ]:
#@title Step 2 — download training data (~30 public-domain books) { display-mode: "form" }
# Classic literature from Project Gutenberg (public domain).
# ADD YOUR OWN: put more IDs in BOOK_IDS, or add any URLs to MY_URLS below.
import urllib.request, os

BOOK_IDS = [11, 12, 74, 76, 84, 98, 120, 158, 161, 174, 215, 244, 345, 408,
            514, 730, 768, 844, 902, 1080, 1184, 1232, 1260, 1342, 1400,
            1661, 1952, 2542, 2591, 2701, 5200, 16389]
MY_URLS = []  # e.g. ["https://en.wikisource.org/wiki/..."] -- pages YOU choose

os.makedirs("my_ai/data/raw", exist_ok=True)
ok = 0
for bid in BOOK_IDS:
    dest = f"my_ai/data/raw/gutenberg_{bid}.txt"
    if os.path.exists(dest): ok += 1; continue
    for url in (f"https://www.gutenberg.org/cache/epub/{bid}/pg{bid}.txt",
                f"https://www.gutenberg.org/files/{bid}/{bid}-0.txt"):
        try:
            txt = urllib.request.urlopen(url, timeout=30).read().decode("utf-8", "replace")
            start = txt.find("*** START"); end = txt.find("*** END")
            if start != -1: txt = txt[txt.find("\n", start):]
            if end != -1: txt = txt[:end]
            open(dest, "w").write(txt); ok += 1; break
        except Exception: continue
print(f"books downloaded: {ok}/{len(BOOK_IDS)}")

if MY_URLS:
    from my_ai.data.ingest_urls import fetch_url
    import hashlib
    for u in MY_URLS:
        try:
            t = fetch_url(u)
            open(f"my_ai/data/raw/web_{hashlib.sha1(u.encode()).hexdigest()[:10]}.txt", "w").write(t)
            print("OK", u)
        except Exception as e: print("SKIP", u, e)

# refresh chat-format examples too (hand-written templates, no AI)
!python -m my_ai.data.make_chat_data
total = sum(os.path.getsize(f"my_ai/data/raw/{f}") for f in os.listdir("my_ai/data/raw"))
print(f"corpus size: {total/1e6:.1f} MB")

In [ ]:
#@title Step 3 — train tokenizer + pack tokens (~10–20 min) { display-mode: "form" }
!python -m my_ai.prepare_data --input my_ai/data/raw --out my_ai/data/processed \
    --vocab-size 4096 --tokenizer-sample-chars 600000

In [ ]:
#@title Step 4 — TRAIN on the GPU (~3 h; re-run anytime to train MORE) { display-mode: "form" }
import os
resume = "--resume my_ai/checkpoints/latest.pt" if os.path.exists("my_ai/checkpoints/latest.pt") else ""
!python -m my_ai.train --config my_ai/configs/small_20m.json \
    --data my_ai/data/processed --steps 20000 --batch-size 32 --lr 6e-4 {resume} \
    --sample-prompt "Once upon a time"
print("\nDone. Re-run this cell to continue training from the checkpoint.")

In [ ]:
#@title Step 5 — chat with your newly trained model { display-mode: "form" }
import sys
sys.path.insert(0, "/content/Project-lmarena")
from my_ai.training.trainer import load_checkpoint, pick_device
from my_ai.tokenizer.tokenizer import load_tokenizer, EOS, USER_TOK, ASSISTANT_TOK
from my_ai.inference.generate import generate
from my_ai.chat.cli import build_prompt_ids

device = pick_device()
model, ckpt = load_checkpoint("my_ai/checkpoints/best.pt", device=device)
model.eval()
tokenizer = load_tokenizer("my_ai/checkpoints/tokenizer.json")
print(f"chatting with step-{ckpt.get('step')} checkpoint. 'quit' to stop.\n")
history = []
while True:
    msg = input("You: ").strip()
    if not msg: continue
    if msg.lower() in ("quit", "exit"): break
    ids = build_prompt_ids(tokenizer, [], history, msg, model.cfg.context_length)
    out = generate(model, ids, max_new_tokens=200, temperature=0.8, top_k=50,
                   top_p=0.95, repetition_penalty=1.15,
                   stop_tokens={EOS, USER_TOK, ASSISTANT_TOK}, device=device)
    reply = tokenizer.decode(out).strip()
    print("AI:", reply, "\n")
    history.append({"role": "user", "text": msg}); history.append({"role": "assistant", "text": reply})

In [ ]:
#@title Step 6 — export for your phone (Termux) and download { display-mode: "form" }
!python -m my_ai.inference.export_numpy --checkpoint my_ai/checkpoints/best.pt \
    --out my_ai/checkpoints/model_numpy.npz
from google.colab import files
files.download("my_ai/checkpoints/model_numpy.npz")
files.download("my_ai/checkpoints/tokenizer.json")
print("Put both files into Project-lmarena/my_ai/checkpoints/ on your phone,")
print("then run:  python -m my_ai.chat.termux_chat")

### Training *more and more*
- **Re-run Step 4** whenever you want — it resumes from `latest.pt` and keeps improving.
- **Add data**: more Gutenberg IDs or your own URLs in Step 2, re-run Steps 2–4.
  ⚠️ If you change the data, delete old checkpoints first if you also changed the
  tokenizer/vocab (`!rm my_ai/checkpoints/*.pt`) — a model is tied to its tokenizer.
- **Save between sessions**: Colab wipes files when the runtime dies. Download
  checkpoints (Step 6) or mount Google Drive and copy `my_ai/checkpoints/` there.
- **Bigger model**: swap `small_20m.json` for `base_50m.json` in Step 4 (slower per step).
- Free Colab gives a few GPU hours per day — train in sessions, resume each time.